# 🐄 Cattle Analytics — Colab Pro+ Experiment Notebook

**Pipeline**: YOLOv8 Detection → ByteTrack Tracking → MobileNetV3 Behavior Classification

**Runtime**: Runtime → Change runtime type → **A100 GPU** (Colab Pro+)

Scripts 01, 02, 03 are **incremental** — they skip already-processed versions automatically.

## 📦 Cell 1 — Install Dependencies
> Run once per session.

In [ ]:
%%capture
!pip install ultralytics==8.3.0
!pip install supervision==0.21.0
!pip install lap
!pip install albumentations
!pip install timm
!pip install seaborn scikit-learn tabulate pyyaml

import torch
print(f'CUDA : {torch.cuda.is_available()}')
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## ☁️ Cell 2 — Mount Drive & Pull Code

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys

DRIVE_ROOT  = '/content/drive/MyDrive/cattle-analytics'
GITHUB_REPO = 'https://github.com/YOUR_ORG/cattle-analytics.git'
CODE_DIR    = f'{DRIVE_ROOT}/cattle-tracker'

os.makedirs(DRIVE_ROOT, exist_ok=True)
if os.path.exists(f'{DRIVE_ROOT}/.git'):
    print('Pulling latest...')
    !cd {DRIVE_ROOT} && git pull
else:
    !git clone {GITHUB_REPO} {DRIVE_ROOT}

sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print(f'Working dir: {os.getcwd()}')

## 🗂️ Cell 3 — Setup Drive Paths & Discover Version Folders

In [ ]:
import scripts.setup_drive as cfg

VERSION_FOLDERS = cfg.VERSION_FOLDERS
FRAMES_DIR      = cfg.FRAMES_DIR
YOLO_DIR        = cfg.YOLO_DIR
CROPS_DIR       = cfg.CROPS_DIR
MODELS_DIR      = cfg.MODELS_DIR
LOGS_DIR        = cfg.LOGS_DIR

print(f'Discovered {len(VERSION_FOLDERS)} version folder(s): {[v.name for v in VERSION_FOLDERS]}')

## 🏞️ Cell 4 — Extract Annotated Frames *(incremental)*
> Skips already-done versions. Use `--force vX` to redo specific ones.

In [ ]:
!python scripts/01_extract_frames.py --width 1280 --height 720
# !python scripts/01_extract_frames.py --force v13 v14   # re-extract specific

## 🏷️ Cell 5 — Convert CVAT XML → YOLO Format *(incremental)*
> Already-done versions keep their split. Only new versions are assigned.

In [ ]:
VAL_VERSIONS = 'v7'   # e.g. 'v7 v14 v21'

!python scripts/02_cvat_to_yolo.py \
    --split_mode manual \
    --val_versions {VAL_VERSIONS}
# !python scripts/02_cvat_to_yolo.py --split_mode auto --val_split 0.2

## ✂️ Cell 6 — Generate Behavior Crops *(incremental)*

In [ ]:
!python scripts/03_generate_crops.py --padding 0.15
# !python scripts/03_generate_crops.py --force v13 v14

## 🔍 Cell 7 — Dataset Health Check
> **Always run before training.** Shows frame counts, behavior distribution, class imbalance warnings.

In [ ]:
!python scripts/04_check_dataset_stats.py

## 🖼️ Cell 8 — Visualize Annotations *(optional)*

In [ ]:
!python scripts/05_visualize_annotations.py --version v1 --max_frames 10

from IPython.display import Image as IPyImage
import glob
for s in glob.glob('processed/viz/v1/*.jpg')[:3]:
    display(IPyImage(s, width=800))

## 🚀 Cell 8b — Copy Dataset to Local NVMe *(run once per session, before training)*
> **Why**: Google Drive throttles at ~3k small file reads. Local NVMe is **100× faster**.
> Copies YOLO frames + behavior crops to `/content/` once (~10–15 min), then all training
> I/O runs at full disk speed. Models and logs still save back to Drive.
>
> **Idempotent**: safe to re-run — skips copy if files already exist this session.
> Training scripts **auto-detect** the local copy and use it automatically.

In [ ]:
import shutil, time, yaml
from pathlib import Path

DRIVE_PROC  = '/content/drive/MyDrive/cattle-analytics/processed'
LOCAL_YOLO  = Path('/content/local_yolo')
LOCAL_CROPS = Path('/content/local_crops')

# ── 1. Copy YOLO dataset (images + labels) ─────────────────────────────────
if LOCAL_YOLO.exists():
    print(f'⏭️  YOLO dataset already at {LOCAL_YOLO} — skipping')
else:
    src = Path(f'{DRIVE_PROC}/yolo_detection')
    print(f'📦 Copying YOLO dataset from Drive...')
    t = time.time()
    shutil.copytree(str(src), str(LOCAL_YOLO))
    print(f'✅ YOLO dataset copied in {(time.time()-t)/60:.1f} min')

# ── 2. Patch dataset.yaml to point to local path ──────────────────────────
# CRITICAL: YOLO reads the 'path:' field from dataset.yaml to locate images/labels.
# After copying, we must update it to /content/local_yolo, not the Drive path.
yaml_path = LOCAL_YOLO / 'dataset.yaml'
with open(yaml_path) as f:
    ydata = yaml.safe_load(f)
ydata['path'] = str(LOCAL_YOLO)
with open(yaml_path, 'w') as f:
    yaml.dump(ydata, f, default_flow_style=False)
print(f'✅ dataset.yaml patched → path = {LOCAL_YOLO}')

# ── 3. Copy behavior crops ──────────────────────────────────────────────
if LOCAL_CROPS.exists():
    print(f'⏭️  Crops already at {LOCAL_CROPS} — skipping')
else:
    src = Path(f'{DRIVE_PROC}/behavior_crops')
    print(f'📦 Copying behavior crops from Drive...')
    t = time.time()
    shutil.copytree(str(src), str(LOCAL_CROPS))
    print(f'✅ Crops copied in {(time.time()-t)/60:.1f} min')

# ── 4. Verify ───────────────────────────────────────────────────────────
n_yolo  = sum(1 for _ in (LOCAL_YOLO / 'images').rglob('*.jpg'))  if (LOCAL_YOLO/'images').exists()  else 0
n_crops = sum(1 for _ in LOCAL_CROPS.rglob('*.jpg'))               if LOCAL_CROPS.exists()             else 0
print(f'\n📊 Local NVMe summary:')
print(f'   YOLO images : {n_yolo:,}')
print(f'   Crop images : {n_crops:,}')
print('\n👍 Ready — training scripts auto-detect and use local copies.')

## 🏷️ Cell 9 — Train YOLOv8 Calf Detector
> Auto-detects local NVMe copy. TF32 + AMP + cuDNN benchmark enabled automatically.
> Metrics → `logs/experiments/detector_<ts>/metrics.json`  
> Weights → `models/checkpoints/detector_<ts>/weights/best.pt`  
> ⏱️ ~30–60 min on A100 for 100 epochs.

In [ ]:
# Smoke test — run first to confirm pipeline works
!python train/train_detector.py \
    --model yolov8s.pt \
    --epochs 30 \
    --batch 16 \
    --imgsz 1280

In [ ]:
# Full training run — uncomment when satisfied with smoke test
# !python train/train_detector.py \
#     --model yolov8s.pt \
#     --epochs 100 \
#     --batch 16 \
#     --imgsz 1280

## 🧠 Cell 10 — Train Behavior Classifier
> Auto-detects local NVMe crops. TF32 + AMP + torch.compile enabled automatically.
> `num_workers=0` required on Colab (Drive-backed loaders deadlock otherwise).  
> ⚠️ Need ≥200 crops per class — check Cell 7 first.  
> ⏱️ ~10–20 min on A100 for 50 epochs.

In [ ]:
!python train/train_classifier.py \
    --epochs 50 \
    --batch 256 \
    --lr 1e-4

## 📊 Cell 11 — Evaluate & Compare All Experiments

In [ ]:
!python evaluation/eval_pipeline.py --phase summary

In [ ]:
!python evaluation/eval_pipeline.py --phase classifier

from IPython.display import Image as IPyImage
import glob
cms = sorted(glob.glob('models/checkpoints/classifier_*/confusion_matrix.png'))
if cms:
    display(IPyImage(cms[-1], width=700))

## 🎦 Cell 12 — Run Full Inference Pipeline on a Video
> Detect → Track → Classify → Annotated video + JSON output

In [ ]:
import glob, os

detectors   = sorted(glob.glob('models/checkpoints/detector_*/weights/best.pt'))
classifiers = sorted(glob.glob('models/checkpoints/classifier_*/best_classifier.pth'))

DETECTOR_PATH   = detectors[-1]   if detectors   else None
CLASSIFIER_PATH = classifiers[-1] if classifiers else None

INPUT_VIDEO  = 'data/v1/<your_video>.mp4'   # UPDATE THIS
OUTPUT_VIDEO = 'data/v1/output_annotated.mp4'

print(f'Detector   : {DETECTOR_PATH}')
print(f'Classifier : {CLASSIFIER_PATH}')
print(f'Input      : {INPUT_VIDEO}')

if DETECTOR_PATH and CLASSIFIER_PATH and os.path.exists(INPUT_VIDEO):
    !python inference/pipeline.py \
        --video        {INPUT_VIDEO} \
        --detector     {DETECTOR_PATH} \
        --classifier   {CLASSIFIER_PATH} \
        --output_video {OUTPUT_VIDEO} \
        --conf 0.35 \
        --classify_every 4
else:
    print('Missing detector, classifier, or input video. Check paths above.')

## ➕ Cell 13 — Add More Videos & Retrain
> 1. Upload new video+XML to `data/vN/` on Drive
> 2. Re-run Cells 3 → 11 — old versions skipped, only new ones processed
> 3. Re-run Cell 8b to copy new data to local NVMe before training

In [ ]:
import importlib
import scripts.setup_drive as cfg
importlib.reload(cfg)

VERSION_FOLDERS = cfg.VERSION_FOLDERS
print(f'Total: {len(VERSION_FOLDERS)} folders: {[v.name for v in VERSION_FOLDERS]}')

done    = [v.name for v in VERSION_FOLDERS if (cfg.FRAMES_DIR / v.name / '.done').exists()]
pending = [v.name for v in VERSION_FOLDERS if v.name not in done]
print(f'Already extracted : {done}')
print(f'Pending extraction: {pending}')

In [ ]:
# Force re-process specific versions (e.g. fixed annotations)
# !python scripts/01_extract_frames.py --force v13 v14
# !python scripts/02_cvat_to_yolo.py   --force v13 v14
# !python scripts/03_generate_crops.py --force v13 v14

# Nuclear option — reprocess everything
# !python scripts/01_extract_frames.py --force
# !python scripts/02_cvat_to_yolo.py   --force
# !python scripts/03_generate_crops.py --force